# Day 23 Sensor Fusion trên Kaggle T4

Chọn Accelerator GPU T4 và bật Internet. Gắn hai dataset
Waymo_Lab23_VInAI20K và weights_Lab23_VinAI20K trong mục Input,
kiểm tra thông tin học viên dưới đây, rồi chọn Run All.
Cell cuối tạo /kaggle/working/submission.zip từ mã, tài liệu và
artifacts thật. Dữ liệu Waymo và weights không vào ZIP.

ZIP là bản đóng gói để đưa lên fork GitHub. Theo SUBMISSION.md,
sau đó lấy commit hash cuối và nộp link repo cùng hash trên LMS.

In [ ]:
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

import torch

STUDENT_NAME = "Nguyen Thanh Binh"
STUDENT_ID = "2A202602777"
STUDENT_EMAIL = 'binhnt1509204@gmail.com'  # Kiểm tra trước khi chạy.
REPO_URL = 'https://github.com/ThanhBinh159/K4-L2L3-DAY23-NguyenThanhBinh-2A202602777-SensorFusion.git'
BASE_COMMIT = 'f6f950c5618f636d54563440d1bd31250e7ae32d'
SEGMENT = 'training_segment-1005081002024129653_5313_150_5333_150_with_camera_labels.tfrecord'
REPO_ROOT = Path("/kaggle/working/day23_sensorfusion_repo")
VENV_ROOT = Path("/tmp/day23_lab_venv")
VENV_PYTHON = VENV_ROOT / "bin/python"
PROTO_ROOT = VENV_ROOT / "protobuf_override"
OUTPUT_ZIP = Path("/kaggle/working/submission.zip")

if sys.version_info < (3, 12):
    raise RuntimeError(f"Lab cần Python >=3.12; hiện tại {sys.version}")
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0).upper():
    raise RuntimeError("Hãy chọn GPU T4 trong Kaggle Settings")
print("Python:", sys.version.split()[0], "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
input_root = Path("/kaggle/input")
if not input_root.is_dir():
    raise FileNotFoundError("Không thấy /kaggle/input")

def one_file(folder, filename):
    matches = list(folder.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Cần đúng một {filename} trong {folder}; tìm thấy {len(matches)}"
        )
    return matches[0]

waymo_file = one_file(input_root, SEGMENT)
weights_file = one_file(input_root, "fpn_resnet_18_epoch_300.pth")
print("Waymo:", waymo_file)
print("Weights:", weights_file)

In [ ]:
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "checkout", "--detach", BASE_COMMIT],
        check=True,
    )
if not (REPO_ROOT / ".git").is_dir():
    raise RuntimeError(f"{REPO_ROOT} tồn tại nhưng không phải git clone")
head = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
if head != BASE_COMMIT:
    raise RuntimeError(f"Repo đang ở commit {head}; notebook cần {BASE_COMMIT}")

config_path = REPO_ROOT / "student/config/paths.yaml"
config_path.write_text(
    f"waymo_dir: {json.dumps(str(waymo_file.parent))}\n"
    f"weights_dir: {json.dumps(str(weights_file.parent))}\n"
    f"segment: {SEGMENT}\nframe_start: 0\nframe_end: 198\n",
    encoding="utf-8",
)
os.environ["DAY23_STUDENT_ROOT"] = str(REPO_ROOT / "student")
os.environ["FUSION_LAB_PLATFORM"] = str(REPO_ROOT / "platform")
os.environ["DAY23_HEADLESS"] = "1"
print("Đã tạo cấu hình:", config_path)

In [ ]:
# Part E
(REPO_ROOT / 'student/workspace/kalman.py').write_text(r'''"""Extended Kalman filter helpers for 6D constant-velocity motion.

Part E supplies prediction and correction for docs/HUONG_DAN_KY_THUAT.md §2.
Read the shared time step and process-noise settings with get_tracking_params().
"""

from __future__ import annotations

from typing import Any
from typing import Optional

import numpy as np

from fusion_lab.workspace_support import get_tracking_params

Matrix = np.matrix | np.ndarray

# vi: Gợi ý module — params = get_tracking_params() sau khi import ở trên.


def build_F(dt: Optional[float] = None) -> Matrix:
    """Build the constant-velocity state transition matrix F.

    Args:
        dt: Time step in seconds; default from tracking params.

    Returns:
        6x6 state transition matrix as ``np.matrix``.
    """
    if dt is None:
        dt = get_tracking_params().dt
    F = np.asmatrix(np.eye(6))
    F[0, 3] = F[1, 4] = F[2, 5] = dt
    return F


def build_Q(dt: Optional[float] = None, q: Optional[float] = None) -> Matrix:
    """Build the process noise covariance matrix Q.

    Args:
        dt: Time step; default from tracking params.
        q: Process noise scale; default from tracking params.

    Returns:
        6x6 process noise matrix.
    """
    params = get_tracking_params()
    scale = (params.dt if dt is None else dt) * (params.q if q is None else q)
    return np.asmatrix(np.eye(6) * scale)


def ekf_predict(
    x: Matrix,
    P: Matrix,
    F: Optional[Matrix] = None,
    Q: Optional[Matrix] = None,
) -> tuple[Matrix, Matrix]:
    """Predict state and covariance one time step forward.

    Args:
        x: State vector (6x1).
        P: State covariance (6x6).
        F: Optional transition matrix; build via ``build_F`` if None.
        Q: Optional process noise; build via ``build_Q`` if None.

    Returns:
        Tuple ``(x_pred, P_pred)``.
    """
    F = build_F() if F is None else np.asmatrix(F)
    Q = build_Q() if Q is None else np.asmatrix(Q)
    x, P = np.asmatrix(x), np.asmatrix(P)
    return F @ x, F @ P @ F.T + Q


def innovation(x: Matrix, meas: Any) -> Matrix:
    """Compute the measurement residual (innovation) gamma.

    Args:
        x: Predicted state.
        meas: Measurement with ``z`` and ``sensor.get_hx(x)``.

    Returns:
        Innovation vector ``z - h(x)``.
    """
    return np.asmatrix(meas.z) - np.asmatrix(meas.sensor.get_hx(x))


def innovation_covariance(P: Matrix, meas: Any, H: Matrix) -> Matrix:
    """Compute the innovation covariance S = H P H' + R.

    Args:
        P: State covariance.
        meas: Measurement with ``R``.
        H: Measurement Jacobian.

    Returns:
        Innovation covariance matrix S.
    """
    H = np.asmatrix(H)
    return H @ np.asmatrix(P) @ H.T + np.asmatrix(meas.R)


def ekf_update(x: Matrix, P: Matrix, meas: Any) -> tuple[Matrix, Matrix]:
    """Apply an EKF measurement update and return updated state and covariance.

    Args:
        x: Prior state.
        P: Prior covariance.
        meas: Associated measurement.

    Returns:
        Tuple ``(x_upd, P_upd)``.
    """
    x, P = np.asmatrix(x), np.asmatrix(P)
    H = np.asmatrix(meas.sensor.get_H(x))
    gamma = innovation(x, meas)
    S = innovation_covariance(P, meas, H)
    K = P @ H.T @ np.linalg.inv(S)
    return x + K @ gamma, (np.asmatrix(np.eye(6)) - K @ H) @ P
''', encoding='utf-8')
print('Đã nạp student/workspace/kalman.py')

In [ ]:
# Part G
(REPO_ROOT / 'student/workspace/camera_fusion.py').write_text(r'''"""Camera field-of-view checks and pinhole measurement modeling.

Part G supplies visibility, projection, and pixel covariance (docs/HUONG_DAN_KY_THUAT.md §2).
The platform differentiates projection using a chain-rule Jacobian.
"""

from __future__ import annotations

from typing import Any
from typing import Sequence

import numpy as np

from fusion_lab.workspace_support import get_tracking_params

Matrix = np.matrix | np.ndarray

# vi: from fusion_lab.workspace_support import get_tracking_params


def is_in_field_of_view(x: Matrix, sensor: Any) -> bool:
    """Return True if state x is visible within the sensor horizontal field of view.

    Args:
        x: State vector (6x1) with position in vehicle frame.
        sensor: Lidar or camera adapter with ``veh_to_sens`` and ``fov``
            (radians).

    Returns:
        True if sensor coordinates are finite and the horizontal angle is within
        ``sensor.fov``. A camera additionally requires depth > 1e-6.
    """
    transform = np.asarray(sensor.veh_to_sens)
    position = transform[:3, :3] @ np.asarray(x).reshape(-1)[:3] + transform[:3, 3]
    if not np.isfinite(position).all() or (
        sensor.name == "camera" and position[0] <= 1e-6
    ):
        return False
    angle = np.arctan2(position[1], position[0])
    return bool(sensor.fov[0] <= angle <= sensor.fov[1])


def camera_measurement_prediction(x: Matrix, sensor: Any) -> Matrix:
    """Predict image-plane measurement h(x) using the pinhole camera model.

    Args:
        x: State vector.
        sensor: Camera with intrinsics ``f_i, f_j, c_i, c_j``.

    Returns:
        2x1 predicted pixel coordinates as ``np.matrix``.

    Raises:
        ValueError: With coordinate context if sensor coordinates are nonfinite
            or depth is at most 1e-6.
    """
    transform = np.asarray(sensor.veh_to_sens)
    position = transform[:3, :3] @ np.asarray(x).reshape(-1)[:3] + transform[:3, 3]
    if not np.isfinite(position).all() or position[0] <= 1e-6:
        raise ValueError(
            "Camera projection needs finite coordinates and positive depth "
            f"> 1e-6; sensor position={position.tolist()}"
        )
    depth, left, up = position
    return np.asmatrix([[sensor.c_i - sensor.f_i * left / depth],
                        [sensor.c_j - sensor.f_j * up / depth]])


def build_camera_measurement(z: Sequence[float], sensor: Any) -> dict[str, Any]:
    """Build camera measurement vector z and covariance R from pixel coordinates.

    Args:
        z: Sequence ``[u, v]`` pixel coordinates.
        sensor: Camera sensor object.

    Returns:
        Dict with keys ``z``, ``R``, ``sensor``.
    """
    params = get_tracking_params()
    return {
        "z": np.asmatrix(np.asarray(z, dtype=float).reshape(2, 1)),
        "R": np.asmatrix(np.diag([params.sigma_cam_i**2, params.sigma_cam_j**2])),
        "sensor": sensor,
    }
''', encoding='utf-8')
print('Đã nạp student/workspace/camera_fusion.py')

In [ ]:
# Part F
(REPO_ROOT / 'student/workspace/association.py').write_text(r'''"""Measurement-to-track association via Mahalanobis gating and greedy matching.

Part F supplies the association stage shown in docs/HUONG_DAN_KY_THUAT.md §2.
Load ``kalman`` with ``load_workspace_module`` for innovation helpers and tracking parameters
for the chi-square gate.
"""

from __future__ import annotations

from typing import Any
from typing import Sequence

import numpy as np
from scipy.stats import chi2

from fusion_lab.workspace_loader import load_workspace_module
from fusion_lab.workspace_support import get_tracking_params

kalman = load_workspace_module("kalman")


def mahalanobis_distance(track: Any, meas: Any) -> float:
    """Return squared Mahalanobis distance between a track and a measurement.

    Args:
        track: Track with ``x``, ``P``.
        meas: Measurement with ``sensor``.

    Returns:
        Scalar squared Mahalanobis distance.
    """
    H = meas.sensor.get_H(track.x)
    gamma = kalman.innovation(track.x, meas)
    S = kalman.innovation_covariance(track.P, meas, H)
    return float((gamma.T @ np.linalg.solve(S, gamma))[0, 0])


def chi2_gate(mhd_sq: float, sensor: Any) -> bool:
    """Return True if squared Mahalanobis distance lies inside the chi-square gate.

    Args:
        mhd_sq: Squared Mahalanobis distance.
        sensor: Sensor with ``dim_meas``.

    Returns:
        True if inside gate.
    """
    threshold = chi2.ppf(get_tracking_params().gating_threshold, sensor.dim_meas)
    return bool(mhd_sq <= threshold)


def association_cost_matrix(
    track_list: Sequence[Any], meas_list: Sequence[Any]
) -> np.matrix:
    """Build gated costs, checking each sensor's visibility before projection.

    Args:
        track_list: Active tracks.
        meas_list: Measurements for this sensor pass.

    Returns:
        Cost matrix; ``np.inf`` for invisible tracks or rejected chi-square gates.
        Invisible pairs must never call the Mahalanobis/projection helpers.
    """
    costs = np.asmatrix(np.full((len(track_list), len(meas_list)), np.inf))
    for i, track in enumerate(track_list):
        for j, meas in enumerate(meas_list):
            if meas.sensor.in_fov(track.x):
                distance = mahalanobis_distance(track, meas)
                if chi2_gate(distance, meas.sensor):
                    costs[i, j] = distance
    return costs


def pick_next_pair(
    association_matrix: np.matrix,
    unassigned_tracks: Sequence[Any],
    unassigned_meas: Sequence[Any],
) -> tuple[Any, Any, np.matrix, list[Any], list[Any]]:
    """Pick the minimum-cost track/measurement pair and shrink the association problem.

    Args:
        association_matrix: Current cost matrix.
        unassigned_tracks: Track objects still free.
        unassigned_meas: Measurement objects still free.

    Returns:
        Tuple (track, meas, new_matrix, remaining_tracks, remaining_meas).
        If no finite pair exists, return np.nan for track and meas and retain both lists.
    """
    if not np.isfinite(association_matrix).any():
        return (
            np.nan, np.nan, association_matrix,
            list(unassigned_tracks), list(unassigned_meas),
        )
    i, j = np.unravel_index(np.argmin(association_matrix), association_matrix.shape)
    tracks, measurements = list(unassigned_tracks), list(unassigned_meas)
    track, meas = tracks.pop(i), measurements.pop(j)
    remaining = np.asmatrix(
        np.delete(np.delete(association_matrix, i, axis=0), j, axis=1)
    )
    return track, meas, remaining, tracks, measurements


def associate_and_update(
    manager: Any,
    meas_list: Sequence[Any],
    filter_obj: Any,
    sensor: Any,
) -> None:
    """Greedy association loop with EKF updates and track management.

    Args:
        manager: Track manager (``track_list``, ``manage_tracks``, ...).
        meas_list: Lidar or camera measurements for this frame pass.
        filter_obj: Filter with ``predict`` / ``update``.
        sensor: Explicit lidar/camera pass sensor, including empty measurement frames.

    Returns:
        None; updates tracks in place and always finishes the lifecycle pass.
        Visibility is handled in the cost matrix, before pair removal. Camera
        updates refine state only; lidar hits alone increase existence scores.
    """
    unassigned_tracks = list(manager.track_list)
    unassigned_meas = list(meas_list)
    costs = association_cost_matrix(unassigned_tracks, unassigned_meas)
    while np.isfinite(costs).any():
        track, meas, costs, unassigned_tracks, unassigned_meas = pick_next_pair(
            costs, unassigned_tracks, unassigned_meas
        )
        filter_obj.update(track, meas)
        manager.handle_updated_track(track, sensor)
    manager.manage_tracks(unassigned_tracks, unassigned_meas, sensor)
''', encoding='utf-8')
print('Đã nạp student/workspace/association.py')

In [ ]:
# Part H
(REPO_ROOT / 'student/workspace/track_management.py').write_text(r'''"""Track initialization, scoring, and deletion helpers.

Part H supplies lidar-driven existence decisions (docs/HUONG_DAN_KY_THUAT.md §2).
Use tracking parameters for the score window, thresholds, and covariance limit.
"""

from __future__ import annotations

from typing import Any

import numpy as np

from fusion_lab.workspace_support import get_tracking_params


def init_track_state_from_meas(meas: Any) -> dict[str, Any]:
    """Initialize track state, covariance, lifecycle state, and score from a measurement.

    Args:
        meas: Lidar measurement with ``z``, ``R``, ``sensor``.

    Returns:
        Dict with keys ``x``, ``P``, ``state``, ``score`` (matrices as ``np.matrix``).
    """
    params = get_tracking_params()
    transform = np.asmatrix(meas.sensor.sens_to_veh)
    rotation = transform[:3, :3]
    x = np.asmatrix(np.zeros((6, 1)))
    x[:3] = rotation @ np.asmatrix(meas.z) + transform[:3, 3]
    P = np.asmatrix(np.zeros((6, 6)))
    P[:3, :3] = rotation @ np.asmatrix(meas.R) @ rotation.T
    P[3, 3], P[4, 4], P[5, 5] = (
        params.sigma_p44**2, params.sigma_p55**2, params.sigma_p66**2
    )
    return {"x": x, "P": P, "state": "initialized", "score": 1 / params.window}


def update_track_score(track: dict[str, Any], associated: bool) -> dict[str, Any]:
    """Update existence once per lidar frame; camera passes never call this helper.

    A hit adds 1/window, capped at one; an in-FOV miss subtracts 1/window.
    Confirm above confirmed_threshold, and preserve confirmed state after misses.

    Args:
        track: Dict-like track with ``score``, ``state``.
        associated: True for a lidar hit; False for a lidar miss within the lidar FOV.

    Returns:
        Updated track dict.
    """
    params = get_tracking_params()
    step = 1 / params.window
    track["score"] = (
        min(1.0, track["score"] + step)
        if associated else track["score"] - step
    )
    if track["score"] > params.confirmed_threshold:
        track["state"] = "confirmed"
    elif associated and track["state"] != "confirmed":
        track["state"] = "tentative"
    return track


def should_delete_track(track: dict[str, Any]) -> bool:
    """Return whether a lidar lifecycle pass should remove this track.

    Delete if either horizontal variance exceeds max_P, or if a confirmed
    track has score < delete_threshold, or an unconfirmed track has score <= 0.
    Camera passes never trigger deletion.

    Args:
        track: Dict with ``score``, ``state``, ``P``.

    Returns:
        True if track should be removed.
    """
    params = get_tracking_params()
    P = track["P"]
    if P[0, 0] > params.max_P or P[1, 1] > params.max_P:
        return True
    if track["state"] == "confirmed":
        return bool(track["score"] < params.delete_threshold)
    return bool(track["score"] <= 0)
''', encoding='utf-8')
print('Đã nạp student/workspace/track_management.py')

In [ ]:
if not VENV_PYTHON.is_file():
    subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", str(VENV_ROOT)],
        check=True,
    )
subprocess.run(
    [
        str(VENV_PYTHON), "-m", "pip", "install", "-q",
        "-e", str(REPO_ROOT / "platform/third_party/waymo_reader"),
        "-e", str(REPO_ROOT / "platform"),
        "-e", str(REPO_ROOT / "student"),
        "pytest",
    ],
    cwd=REPO_ROOT,
    check=True,
)
subprocess.run(
    [
        str(VENV_PYTHON), "-m", "pip", "install", "-q", "--upgrade",
        "--ignore-installed",
        "--no-deps", "--target", str(PROTO_ROOT), "protobuf>=6.33.5,<7",
    ],
    check=True,
)
if not (PROTO_ROOT / "google/protobuf/__init__.py").is_file():
    raise RuntimeError(f"pip chưa cài protobuf vào {PROTO_ROOT}")
PROTO_BOOTSTRAP = (
    "import google\n"
    f"google.__path__ = [{str(PROTO_ROOT / 'google')!r}, *google.__path__]\n"
    "import google.protobuf\n"
    "assert google.protobuf.__version__.split('.')[0] == '6'\n"
)
subprocess.run(
    [
        str(VENV_PYTHON), "-c",
        PROTO_BOOTSTRAP + (
            "import torch; "
            "print('protobuf:', google.protobuf.__version__, google.protobuf.__file__); "
            "from simple_waymo_open_dataset_reader import dataset_pb2; "
            "assert torch.cuda.is_available(); "
            "print('torch GPU:', torch.cuda.get_device_name(0))"
        ),
    ],
    check=True,
)

In [ ]:
result = subprocess.run(
    [
        str(VENV_PYTHON), "-c",
        PROTO_BOOTSTRAP + "import pytest; raise SystemExit(pytest.main(['student/tests', '-q']))",
    ],
    cwd=REPO_ROOT, text=True, capture_output=True,
)
print(result.stdout[-6000:])
if result.returncode:
    print(result.stderr[-3000:])
    result.check_returncode()
if "xfailed" in result.stdout.lower():
    raise RuntimeError("Còn test E-H xfailed")

## Compare trên T4

Part C gốc mặc định chạy CPU. Cell sau chuyển model và BEV tensor
sang CUDA trong bộ nhớ tiến trình chạy. File Part C đóng gói vẫn
giữ nguyên. Runner sinh sáu artifacts chuẩn.

In [ ]:
gpu_runner = r'''
import sys
from pathlib import Path
import torch
from fusion_lab.scripts import run_lab
from fusion_lab.workspace_loader import load_workspace as original_loader

if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable")

def load_gpu_workspace():
    workspace = original_loader()
    detector = workspace["detection_pipeline"]
    original_config = detector.load_fpn_resnet_config
    original_detect = detector.detect_objects_from_bev

    def gpu_config(*args, **kwargs):
        config = original_config(*args, **kwargs)
        config.device = torch.device("cuda:0")
        config.no_cuda = False
        return config

    def gpu_detect(bev, model, config):
        if not next(model.parameters()).is_cuda:
            raise RuntimeError("Detector model did not move to GPU")
        return original_detect(bev.to(config.device), model, config)

    detector.load_fpn_resnet_config = gpu_config
    detector.detect_objects_from_bev = gpu_detect
    return workspace

run_lab.load_workspace = load_gpu_workspace
metrics = run_lab.run(Path(sys.argv[1]), "compare", seed=0)
assert metrics["fusion_mode"] == "compare"
assert metrics["frames"] == [0, 198]
assert metrics["seed"] == 0
assert metrics["tracking"]["lidar"] is not None
assert metrics["tracking"]["fused"] is not None
'''
subprocess.run(
    [str(VENV_PYTHON), "-c", PROTO_BOOTSTRAP + gpu_runner, str(config_path)],
    cwd=REPO_ROOT, env=os.environ.copy(), check=True,
)

In [ ]:
artifacts = REPO_ROOT / "student/artifacts"
metrics = json.loads((artifacts / "metrics.json").read_text(encoding="utf-8"))
if metrics["fusion_mode"] != "compare" or metrics["frames"] != [0, 198] or metrics["seed"] != 0:
    raise ValueError("Lần chấm không đủ compare, seed 0, frames 0-198")
if metrics["segment"] != SEGMENT:
    raise ValueError(f"Segment không đúng: {metrics['segment']}")
detection = metrics["detection"]
lidar = metrics["tracking"]["lidar"]
fused = metrics["tracking"]["fused"]
if lidar is None or fused is None:
    raise ValueError("Cần cả tracking.lidar và tracking.fused")

def summary(result):
    return (
        f"rmse={result['rmse']} m; matches={result['matches']}; "
        f"sum_sq_err={result['sum_sq_err']} m²; "
        f"ghost_track_frames={result['ghost_track_frames']}; "
        f"missed_gt_frames={result['missed_gt_frames']}; "
        f"mean_confirmed_tracks={result['mean_confirmed_tracks']}"
    )

delta_rmse = (
    "không xác định" if lidar["rmse"] is None or fused["rmse"] is None
    else f"{fused['rmse'] - lidar['rmse']:+.4f} m"
)
lidar_denominator = lidar["matches"] + lidar["ghost_track_frames"]
fused_denominator = fused["matches"] + fused["ghost_track_frames"]
lidar_precision = lidar["matches"] / lidar_denominator if lidar_denominator else 0.0
fused_precision = fused["matches"] / fused_denominator if fused_denominator else 0.0
lidar_coverage = lidar["matches"] / detection["tp"] if detection["tp"] else 0.0
fused_coverage = fused["matches"] / detection["tp"] if detection["tp"] else 0.0
fusion_consistent = (
    lidar["rmse"] is not None and fused["rmse"] is not None
    and fused["rmse"] - lidar["rmse"] <= 0.05
)
report = f"""# Báo cáo bài nộp — Day 23 Sensor Fusion Lab

## Thông tin học viên

- Họ tên: {STUDENT_NAME}
- MSSV: {STUDENT_ID}
- Email: {STUDENT_EMAIL}
- Link repo (fork): {REPO_URL.removesuffix('.git')}
- Commit hash nộp (git rev-parse HEAD): Lấy từ commit cuối và nộp trên LMS sau khi push.

## Tóm tắt kết quả

- fusion_mode: {metrics['fusion_mode']}; frames: {metrics['frames']}; segment: {metrics['segment']}; seed: {metrics['seed']}
- detection.precision: {detection['precision']}; detection.recall: {detection['recall']}; tp/fp/fn: {detection['tp']}/{detection['fp']}/{detection['fn']}
- tracking.lidar: {summary(lidar)}
- tracking.fused: {summary(fused)}
- precision_track: LiDAR {lidar_precision:.4f}, fused {fused_precision:.4f}; coverage: LiDAR {lidar_coverage:.4f}, fused {fused_coverage:.4f}.
- So sánh: RMSE fused trừ LiDAR = {delta_rmse}; matches thay đổi {fused['matches'] - lidar['matches']:+d}; ghost thay đổi {fused['ghost_track_frames'] - lidar['ghost_track_frames']:+d}; miss thay đổi {fused['missed_gt_frames'] - lidar['missed_gt_frames']:+d}. RMSE chỉ tính trên các cặp ghép nên cần đọc cùng matches và ghost/miss.
- Tiêu chí RMSE fused không kém LiDAR quá 0.05 m: {"đạt" if fusion_consistent else "chưa đạt"}.
- Giới hạn: camera dùng tâm hộp 2D nhãn FRONT thêm nhiễu theo seed; không có detector ảnh độc lập.
- Môi trường: notebook chuyển model và BEV tensor của Part C sang Kaggle T4 trong bộ nhớ; file Part C nộp giữ nguyên nên khi chạy lại trên CPU có thể khác nhẹ do số học.

## Giải thích ngắn (Parts E–H)

1. LiDAR đo z=(x,y,z) theo mét với R 3×3; camera đo z=(u,v) theo pixel với R 2×2. EKF dùng h(x) và H riêng cho từng sensor để tính innovation và update.
2. Mahalanobis so innovation với S=HPHᵀ+R; cổng χ² theo số chiều đo loại cặp khó tin trước gán greedy. Khoảng cách này có xét độ bất định, còn Euclidean thì không.
3. Lab là track-then-fuse: một danh sách track, predict một lần/frame, AssocL update LiDAR và quản lý track, rồi AssocC update camera. Thứ tự trong platform/fusion_lab/scripts/run_lab.py; grade_run.log ghi record lidar/fused theo frame.
4. Calibration lệch làm pixel dự báo h(x) lệch và innovation tăng. Gating có thể loại đo hoặc update sai nếu độ lệch lọt cổng. Đây là hệ quả mô hình, chưa phải kết quả bonus calibration.
5. Sensor cần truyền tường minh vì frame không có measurement không thể suy ra modality. LiDAR quyết định score/init/delete; camera chỉ update EKF.
6. Xác nhận khi score > 0.8; miss không hạ confirmed. Xóa khi Pxx hoặc Pyy > 9, confirmed có score < 0.6, hoặc track chưa confirmed có score ≤ 0.

## Bonus

Không.

## Khai báo sử dụng AI

- Công cụ đã dùng (ChatGPT, Copilot, Claude, …): ChatGPT/Codex.
- Dùng cho phần nào (hàm, câu hỏi, debug): Part E–H, notebook Kaggle, bản nháp giải thích và báo cáo số liệu.
- Cách bạn đã kiểm tra lại (pytest, chạy Waymo, đối chiếu công thức): Notebook đã chạy pytest student/tests -q, chạy compare seed 0 trên segment mặc định và runner đã đối chiếu metrics.json với grade_run.log. Học viên cần đọc lại code và báo cáo trước khi nộp.

## Checklist nộp

- [x] Part E–H đã implement và test trong repo pass khi chạy notebook
- [x] Compare seed 0, frame 0–198 và sáu artifacts do runner sinh
- [x] ZIP không chứa Waymo, weights, paths.yaml hay .env
- [ ] Đưa nội dung ZIP lên fork GitHub và chạy tools/check_submission.py
- [ ] Nộp link repo và commit hash trên LMS
"""
report = "\n".join(line.lstrip() for line in report.splitlines()) + "\n"
(REPO_ROOT / "student/SUBMISSION.md").write_text(report, encoding="utf-8")

artifacts_required = {
    f"student/artifacts/{name}" for name in (
        "metrics.json", "grade_run.log", "metrics_lidar.json",
        "metrics_fused.json", "grade_run_lidar.log", "grade_run_fused.log"
    )
}
docs_required = {
    "README.md", "CHECKPOINTS.md", "SUBMISSION.md", "RUBRIC.md",
    "RULES.md", "NOTICE.md", "docs/HUONG_DAN_KY_THUAT.md",
    "data/README.md", "student/SUBMISSION.md",
}
code_required = {
    "student/workspace/kalman.py", "student/workspace/association.py",
    "student/workspace/camera_fusion.py", "student/workspace/track_management.py",
}
tracked = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "ls-files", "-z"]
).decode("utf-8").strip("\0").split("\0")
package_files = sorted(set(tracked) | artifacts_required)
required = docs_required | code_required | artifacts_required
if not required.issubset(package_files):
    raise FileNotFoundError(f"ZIP thiếu: {sorted(required - set(package_files))}")
forbidden = (".tfrecord", ".pth", ".pt", ".ckpt", ".pickle", ".zip", ".tar", ".gz", ".7z", ".rar")
for relative in package_files:
    file = REPO_ROOT / relative
    if not file.is_file():
        raise FileNotFoundError(file)
    if file.name in (".env", "paths.yaml") or file.suffix.lower() in forbidden:
        raise ValueError(f"File cấm: {relative}")
    if file.stat().st_size > 20 * 1024 * 1024:
        raise ValueError(f"File lớn hơn 20 MB: {relative}")
with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for relative in package_files:
        archive.write(REPO_ROOT / relative, relative)
with zipfile.ZipFile(OUTPUT_ZIP) as archive:
    if archive.testzip() is not None or not required.issubset(archive.namelist()):
        raise ValueError("ZIP lỗi hoặc thiếu file bắt buộc")
print(f"Đã tạo {OUTPUT_ZIP} ({OUTPUT_ZIP.stat().st_size / 1024**2:.2f} MB)")
print("Markdown trong ZIP:", *sorted(p for p in package_files if p.endswith(".md")), sep="\n- ")